# Model hubs

In this expirement we are going to simulate what the model hubs face when they have to deal with a large number of models and users.

In [1]:
from config import MODEL_SUITE
from neuralzip.compressor import Compressor

JS_THRESHOLD = 0.1

 Let's define the compressors:

In [2]:
# Global Huffman baseline
global_huffman = Compressor(
    "global-huffman",
    verify=False,
)

# ZipNN baseline
zipnn_local = Compressor(
    "zipnn-local", 
    verify=False
)

# NeuralZip-Fast
neuralzip_fast = Compressor(
    "neuralzip-fast",
    verify=False,
    clustering="js-distance",
    cluster_js_threshold=0,
    test_cluster_compressibility=True,
)

# NeuralZip-Ratio
neuralzip_ratio = Compressor(
    "neuralzip-ratio",
    verify=False,
    cluster_js_threshold=JS_THRESHOLD,
)

## Model Storing

The first task we want to test is the storing of models. We want to see how well the compressors can compress a large number of models and how much time it takes to compress them.

In [3]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import time

import pandas as pd
from IPython.display import display

from neuralzip import HFModel

MODEL_DIR = Path("models")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path("outputs/model_hubs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_storing_compressors = [
    ("Global Huffman", global_huffman),
    ("ZipNN local", zipnn_local),
    ("NeuralZip-Fast", neuralzip_fast),
    ("NeuralZip-Ratio", neuralzip_ratio),
]

def _model_source(model_config):
    local_subdir = model_config.get("local_subdir")
    if local_subdir:
        local_path = MODEL_DIR / local_subdir
        if local_path.exists():
            return str(local_path), {"local_files_only": True}
    load_kwargs = dict(model_config.get("load_kwargs", {}))
    load_kwargs.setdefault("local_files_only", True)
    return model_config["model_id"], load_kwargs

def _result_row(model_config, compressor_name, result, *, status="ok", error=None):
    raw_bytes = result.get("raw_bytes") if result else None
    compressed_bytes = result.get("compressed_bytes") if result else None
    compression_ratio = result.get("compression_ratio") if result else None
    return {
        "model": model_config["name"],
        "model_id": model_config["model_id"],
        "parameters_b": model_config.get("parameters_b"),
        "dtype": model_config.get("default_dtype"),
        "compressor": compressor_name,
        "method": result.get("method") if result else None,
        "scope": result.get("scope", "full checkpoint") if result else None,
        "status": status,
        "raw_bytes": raw_bytes,
        "compressed_bytes": compressed_bytes,
        "compression_ratio": compression_ratio,
        "compressed_pct": (100 * compression_ratio) if compression_ratio is not None else None,
        "precompute_seconds": result.get("precompute_seconds") if result else None,
        "encode_seconds": result.get("encode_seconds") if result else None,
        "cold_encode_seconds": result.get("cold_encode_seconds") if result else None,
        "decode_seconds": result.get("decode_seconds") if result else None,
        "clusters": result.get("clusters") if result else None,
        "codebooks": len(result.get("codebooks", ())) if result and result.get("codebooks") is not None else None,
        "huffman_chunks": result.get("huffman_chunks") if result else None,
        "raw_chunks": result.get("raw_chunks") if result else None,
        "roundtrip_ok": result.get("roundtrip_ok") if result else None,
        "error": error,
    }

model_storing_results = []
for model_config in MODEL_SUITE:
    source, load_kwargs = _model_source(model_config)
    try:
        load_start = time.perf_counter()
        hf_model = HFModel(source, dtype=model_config["default_dtype"], **load_kwargs)
        load_seconds = time.perf_counter() - load_start
    except Exception as exc:
        for compressor_name, _ in model_storing_compressors:
            row = _result_row(
                model_config,
                compressor_name,
                None,
                status="load-failed",
                error=f"{type(exc).__name__}: {exc}",
            )
            row["load_seconds"] = None
            model_storing_results.append(row)
        pd.DataFrame(model_storing_results).to_csv(OUTPUT_DIR / "model_storing_results.csv", index=False)
        continue

    for compressor_name, compressor in model_storing_compressors:
        try:
            result = compressor.compress(hf_model)
            row = _result_row(model_config, compressor_name, result)
            row["load_seconds"] = load_seconds
        except Exception as exc:
            row = _result_row(
                model_config,
                compressor_name,
                None,
                status="compress-failed",
                error=f"{type(exc).__name__}: {exc}",
            )
            row["load_seconds"] = load_seconds
        model_storing_results.append(row)
        pd.DataFrame(model_storing_results).to_csv(OUTPUT_DIR / "model_storing_results.csv", index=False)

    del hf_model
    gc.collect()

model_storing_df = pd.DataFrame(model_storing_results)
model_storing_df.to_json(OUTPUT_DIR / "model_storing_results.json", orient="records", indent=2)
display(model_storing_df)


/home/mbravo/bachelor-thesis-code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Loading weights:  56%|█████▌    | 668/1188 [00:00<00:00, 6652.50it/s]

Loading weights: 100%|██████████| 1188/1188 [00:00<00:00, 4843.78it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

...

Loading weights:  65%|██████▍   | 764/1183 [00:00<00:00, 3414.35it/s]

Loading weights:  94%|█████████▎| 1108/1183 [00:00<00:00, 3377.54it/s]

Loading weights: 100%|██████████| 1183/1183 [00:00<00:00, 3412.10it/s]


[transformers] Qwen3_5Model LOAD REPORT from: Qwen/Qwen3.5-27B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,model_id,parameters_b,dtype,compressor,method,scope,status,raw_bytes,compressed_bytes,...,encode_seconds,cold_encode_seconds,decode_seconds,clusters,codebooks,huffman_chunks,raw_chunks,roundtrip_ok,error,load_seconds
0,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,70.0,bf16,Global Huffman,NaN,NaN,load-failed,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,OSError: We couldn't connect to 'https://huggi...,NaN
1,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,70.0,bf16,ZipNN local,NaN,NaN,load-failed,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,OSError: We couldn't connect to 'https://huggi...,NaN
2,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,70.0,bf16,NeuralZip-Fast,NaN,NaN,load-failed,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,OSError: We couldn't connect to 'https://huggi...,NaN
3,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,70.0,bf16,NeuralZip-Ratio,NaN,NaN,load-failed,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,OSError: We couldn't connect to 'https://huggi...,NaN
4,Gemma 4 31B,google/gemma-4-31B,31.0,bf16,Global Huffman,global-huffman,exponent-stream,ok,3.127309e+10,1.685468e+10,...,0.000000,353.370617,None,NaN,NaN,119458.0,466.0,None,NaN,0.970037
5,Gemma 4 31B,google/gemma-4-31B,31.0,bf16,ZipNN local,zipnn-local,full checkpoint,ok,6.254617e+10,4.156483e+10,...,349.609912,349.609912,None,0.0,0.0,239322.0,239110.0,None,NaN,0.970037
6,Gemma 4 31B,google/gemma-4-31B,31.0,bf16,NeuralZip-Fast,neuralzip-fast,full checkpoint,ok,6.254617e+10,4.200443e+10,...,285.254481,528.691359,None,958.0,1916.0,234357.0,244075.0,None,NaN,0.970037
7,Gemma 4 31B,google/gemma-4-31B,31.0,bf16,NeuralZip-Ratio,neuralzip-ratio,bf16-exponent-stream,ok,3.127309e+10,1.013187e+10,...,0.000000,2796.142001,None,257.0,NaN,NaN,NaN,None,NaN,0.970037
8,Phi-4 14B,microsoft/phi-4,14.0,bf16,Global Huffman,global-huffman,exponent-stream,ok,1.414570e+10,4.636859e+09,...,0.000000,131.075410,None,NaN,NaN,53961.0,80.0,None,NaN,0.116607
9,Phi-4 14B,microsoft/phi-4,14.0,bf16,ZipNN local,zipnn-local,full checkpoint,ok,2.829141e+10,1.878071e+10,...,164.772620,164.772620,None,0.0,0.0,108082.0,107920.0,None,NaN,0.116607


## Download

We simulate a download pipeline, this is:

$$
\text{Request} \rightarrow \text{Download} \rightarrow \text{Decompress}
$$

For this our setup is a server that has the compressed model,

## Upload

We simulate a download pipeline, this is:

$$
\text{Request} \rightarrow \text{Compress} \rightarrow \text{Upload}
$$

For this our setup is a server that has the compressed model,